# 13주차 숙제2

In [1]:
from urllib.request import urlretrieve
import pandas as pd

urlretrieve('https://bit.ly/3q9SZix', '20s_best_book.json')
books_df = pd.read_json('20s_best_book.json')
books = books_df.loc[:, 'no':'isbn13']

print(books.head())
print(books_df.loc[[0, 1], ['bookname', 'authors']])
print(books_df.loc[0:1, 'bookname':'authors'])
print(books_df.loc[::2, 'no':'isbn13'].head())

   no  ranking                         bookname      authors publisher  publication_year         isbn13
0   1        1  우리가 빛의 속도로 갈 수 없다면 :김초엽 소설   지은이: 김초엽      허블              2019  9791190090018
1   2        2         달러구트 꿈 백화점.이미예 장편소설   지은이: 이미예      팩토리나인          2020  9791165341909
...


In [2]:
import requests
from bs4 import BeautifulSoup

isbn = 9791190090018
headers = {'User-Agent': 'Mozilla/5.0'}
url = 'https://www.yes24.com/product/search?domain=BOOK&query={}'
r = requests.get(url.format(isbn), headers=headers, timeout=20)
soup = BeautifulSoup(r.text, 'html.parser')
prd_link = soup.find('a', attrs={'class': 'gd_name'})
print(prd_link['href'])

detail_url = 'https://www.yes24.com' + prd_link['href']
r = requests.get(detail_url, headers=headers, timeout=20)
soup = BeautifulSoup(r.text, 'html.parser')
prd_detail = soup.find('div', attrs={'id': 'infoset_specific'})
for tr in prd_detail.find_all('tr'):
    th = tr.find('th')
    if th and th.get_text(strip=True) == '쪽수, 무게, 크기':
        size_info = tr.find('td').get_text(' ', strip=True)
        break
print(size_info)
print([part.strip() for part in size_info.split('|')][1])

/product/goods/74261416
344쪽 | 496g | 130*198*30mm
496g


In [3]:
def get_weight(isbn):
    search_url = 'https://www.yes24.com/product/search?domain=BOOK&query={}'
    response = requests.get(search_url.format(isbn), headers=headers, timeout=20)
    soup = BeautifulSoup(response.text, 'html.parser')
    prd_info = soup.find('a', attrs={'class': 'gd_name'})
    if prd_info is None:
        return ''
    response = requests.get('https://www.yes24.com' + prd_info['href'], headers=headers, timeout=20)
    soup = BeautifulSoup(response.text, 'html.parser')
    prd_detail = soup.find('div', attrs={'id': 'infoset_specific'})
    if prd_detail is None:
        return ''
    for tr in prd_detail.find_all('tr'):
        th = tr.find('th')
        if th and th.get_text(strip=True) == '쪽수, 무게, 크기':
            parts = [part.strip() for part in tr.find('td').get_text(' ', strip=True).split('|')]
            return parts[1] if len(parts) > 1 else ''
    return ''

top10_books = books.head(10)
weight = top10_books.apply(lambda row: get_weight(row['isbn13']), axis=1)
weight.name = 'weight'
print(weight)
top10_with_weight = pd.merge(top10_books, weight, left_index=True, right_index=True)
print(top10_with_weight)

0    496g
1    358g
2    296g
3    412g
4    388g
5    512g
6    318g
7        
8    324g
9    406g
Name: weight, dtype: object
   no  ranking  ...         isbn13 weight
0   1        1  ...  9791190090018   496g
1   2        2  ...  9791165341909   358g
...
[10 rows x 8 columns]
